In [1]:
import numpy as np
from numpy.random import choice
import pandas as pd

from prettytable import PrettyTable
from prettytable import TableStyle

In [2]:
dbImages = np.load("moscow_dbImages.npy", allow_pickle=True)
qImages = np.load("moscow_qImages.npy", allow_pickle=True)
pIdx = np.load("moscow_pIdx.npy", allow_pickle=True)
qIdx = np.load("moscow_qIdx.npy", allow_pickle=True)

# Note: there is no overlap between db images and query images
np.intersect1d(dbImages, qImages)

array([], dtype='<U110')

In [6]:
dbImages

array(['train_val/moscow/database/images/4kD4HurLTvbDVGFwQ3ejtv.jpg',
       'train_val/moscow/database/images/9MoW7UPmhV1P8m17kblIlc.jpg',
       'train_val/moscow/database/images/HJmHtStGeecYSOJ0VcsaS3.jpg', ...,
       'train_val/moscow/database/images/N5FlwIfA0bgCYN_5oJnyW3.jpg',
       'train_val/moscow/database/images/Cp0VqoIITkmgeMlGIS9qNc.jpg',
       'train_val/moscow/database/images/pYamYXYLYXT8ifEnzwJE9h.jpg'],
      shape=(171665,), dtype='<U110')

In [4]:
pIdx

array([array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
              17, 18, 19, 20])                                                   ,
       array([ 21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32, 101,
              102, 103, 104, 105, 106, 107, 108, 109, 110, 111])              ,
       array([ 53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
               66,  67,  68,  69,  70,  71,  72,  73,  74, 119, 120, 121, 122,
              123, 124, 125, 126])                                            ,
       ...,
       array([171539, 171540, 171541, 171542, 171543, 171544, 171545, 171546,
              171547, 171548, 171549, 171550, 171551, 171552, 171553, 171554,
              171555, 171556, 171557, 171558])                               ,
       array([171562, 171563, 171564, 171565, 171566, 171567, 171568, 171569,
              171570, 171571, 171572, 171573, 171574, 171579, 171580, 171581,
              171582, 171583, 17158

### Truncate half of reference images

In [5]:
ratio_choice = lambda x: choice(x, int(len(x) * 0.5), replace=False)
trunc_choice = lambda x: x[3:]
mod_pIdx = np.array(list((map(ratio_choice, pIdx))), dtype="object")
mod_pIdx

array([array([14,  4, 10, 18,  2,  7,  3, 15,  9,  5]),
       array([101, 106,  27,  23, 102,  24,  30,  28,  21,  32, 103]),
       array([ 62,  59,  69,  64,  57, 120, 126,  71,  63,  74,  73,  61,  58,
              123,  66])                                                      ,
       ...,
       array([171554, 171539, 171548, 171546, 171550, 171558, 171555, 171557,
              171543, 171553])                                               ,
       array([171633, 171604, 171644, 171573, 171603, 171643, 171566, 171579,
              171572, 171602, 171567, 171582, 171599, 171640, 171583, 171570,
              171615, 171631, 171564, 171606, 171562, 171635, 171621, 171641,
              171624, 171598, 171580, 171600, 171623, 171622])               ,
       array([171657, 171659, 171658, 171663, 171645, 171662, 171648, 171656,
              171664])                                                       ],
      shape=(3087,), dtype=object)

In [7]:
mod_db_idx = np.intersect1d(np.arange(dbImages.shape[0]), np.hstack(mod_pIdx.flatten()))
mod_db_idx

array([     2,      3,      4, ..., 171662, 171663, 171664],
      shape=(39207,))

In [8]:
mod_dbImages= dbImages[mod_db_idx]
mod_dbImages

array(['train_val/moscow/database/images/HJmHtStGeecYSOJ0VcsaS3.jpg',
       'train_val/moscow/database/images/qLrFa0LuuenqQ3aML3L8BX.jpg',
       'train_val/moscow/database/images/3UfAkX76lVAn3ati74Zv7J.jpg', ...,
       'train_val/moscow/database/images/N5FlwIfA0bgCYN_5oJnyW3.jpg',
       'train_val/moscow/database/images/Cp0VqoIITkmgeMlGIS9qNc.jpg',
       'train_val/moscow/database/images/pYamYXYLYXT8ifEnzwJE9h.jpg'],
      shape=(39207,), dtype='<U110')

### Statistics

In [9]:
def print_count_values(columns: list[str]) -> None:
    for col in columns:
        table = PrettyTable()
        table.set_style(TableStyle.MARKDOWN)
        table.field_names = [col, "database", "query"]
        
        q = pd.read_csv("train_val/moscow/query/postprocessed.csv")[col].value_counts()
        db  = pd.read_csv("train_val/moscow/database/postprocessed.csv")[col].value_counts()
        for i in range(len(q.index)):
            table.add_row([q.index[i], db.values[i], q.values[i]])
        print(table, end=2*"\n")

def print_npy_stats(dbImages, qImages, pIdx) -> None:
    table = PrettyTable()
    table.set_style(TableStyle.MARKDOWN)
    table.field_names = ["Stats", "Cnt images"]
    
    pIdxSeries = pd.Series(pIdx)
    table.add_rows([
        ["dbImages", dbImages.shape[0]],
        ["qImages", qImages.shape[0]],
        ["pIdx", pIdx.shape[0]],
        ["mean", int(pIdxSeries.apply(lambda x: x.shape[0]).mean())],
        ["median", pIdxSeries.apply(lambda x: x.shape[0]).median()],
    ])
    print(table, end=2*"\n")

def print_cnt_captured_at() -> None:
    table = PrettyTable()
    table.set_style(TableStyle.MARKDOWN)
    table.field_names = ["Year", "Cnt images"]

    raw = pd.read_csv("train_val/moscow/database/raw.csv")
    raw["captured_at"] = raw["captured_at"].apply(lambda x: x.split("-")[0])
    captured = raw["captured_at"].value_counts()
    
    for i in range(len(captured.index)):
        table.add_row([captured.index[i], captured.values[i]])
    print(table, end=2*"\n")

print_npy_stats(mod_dbImages, qImages, mod_pIdx)
# print_count_values(["view_direction", "night"])
# print_cnt_captured_at()

|  Stats   | Cnt images |
| :------: | :--------: |
| dbImages |   39207    |
| qImages  |    3103    |
|   pIdx   |    3087    |
|   mean   |     13     |
|  median  |    10.0    |



### Save modified npy files

In [10]:
np.save("modified_moscow_dbImages.npy", mod_dbImages, allow_pickle=True)

In [11]:
np.save("modified_moscow_pIdx.npy", mod_pIdx, allow_pickle=True)